# 08 · Estimaciones con municipios vecinos para la app

**Objetivo:** estimar el indicador de agua subterránea (mal estado cuantitativo) de los **722 municipios sin masa de agua catalogada**, con la media de los 5 municipios más cercanos con dato, y guardar el resultado en `agua_estimada_vecinos_v01.csv`. Ese fichero lo lee el notebook 09 para preparar los datos de la app.

**Orden:** después del 07. Necesita el dataset `sitewise_v03` (01, 02 y 03) y los ficheros del perfil del 07, para comprobar que la app y el perfil estiman los **mismos** municipios.

El cálculo está en `src/sitewise/ml/indices.py` (`tabla_estimaciones_vecinos`), el mismo método que usa el perfil v02 para completar la dimensión hídrica (notebooks 05, 06 y 07), así que el dato estimado es el mismo en la app y en el perfil.

> Este notebook sustituye al antiguo 08 (segunda prueba de clustering con 7 variables, v01) y a la parte de estimación que tenía. Esa versión sigue en el historial de Git (commit `bc30250`).

**Importante:** los valores estimados **no son datos reales**. La app los muestra como `aproximado` y con la distancia a los vecinos usados (lo hace el 09).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))  # por si no se ha hecho pip install -e .

from sitewise.ml.indices import COL_ACUIFERO, VECINOS, tabla_estimaciones_vecinos

DATASET = ROOT / 'data/processed/sitewise_v03/dataset_municipal.csv'
SALIDA = ROOT / 'data/processed/sitewise_ml'
PERFIL_ESTADO = ROOT / 'data/reference/perfiles_estado_v02.csv'

# Comprobación de orden: el 07 debe haberse ejecutado.
if not PERFIL_ESTADO.exists():
    raise FileNotFoundError('Falta perfiles_estado_v02.csv: ejecuta antes el notebook 07.')
pd.set_option('display.width', 200)

d = pd.read_csv(DATASET, dtype={'cod_ine': str})
assert len(d) == 8132 and d['cod_ine'].is_unique
print('Municipios sin dato de agua:', int(d[COL_ACUIFERO].isna().sum()), 'de', len(d))

Municipios sin dato de agua: 722 de 8132


## 1. Estimar y guardar

In [2]:
est = tabla_estimaciones_vecinos(d, COL_ACUIFERO, VECINOS)

# Comprobaciones: son las mismas que exige el notebook 09 al leer el fichero.
assert len(est) == 722 and est['cod_ine'].is_unique and est['cod_ine'].str.fullmatch(r'\d{5}').all()
assert est['valor_estimado_pct'].between(0, 100).all()
assert (est['distancia_maxima_km'] >= est['distancia_media_km']).all()
assert est['k_vecinos'].eq(VECINOS).all() and est['estado_propuesto'].eq('aproximado').all()

# La app y el perfil v02 deben estimar exactamente los mismos municipios (lo marca el 07 en perfiles_estado_v02.csv).
estado = pd.read_csv(PERFIL_ESTADO, dtype={'cod_ine': str})
marcados = set(estado.loc[estado['acuifero_estimado_vecinos'], 'cod_ine'])
assert set(est['cod_ine']) == marcados, 'La app y el perfil v02 no estiman los mismos municipios.'
print('Coinciden con los municipios marcados como estimados en el perfil v02:', len(marcados))

SALIDA.mkdir(parents=True, exist_ok=True)
est.to_csv(SALIDA / 'agua_estimada_vecinos_v01.csv', index=False)
print('Agua estimada para', len(est), 'municipios | media', est['valor_estimado_pct'].mean().round(1), '%')
print('Distancia media a los vecinos (km): mediana', est['distancia_media_km'].median(), '| máxima', est['distancia_media_km'].max())

Coinciden con los municipios marcados como estimados en el perfil v02: 722
Agua estimada para 722 municipios | media 5.6 %
Distancia media a los vecinos (km): mediana 11.75 | máxima 52.9


## 2. ¿Cuánto se puede fiar uno de la estimación?

Con los municipios que **sí** tienen dato se borra el valor de cada uno y se estima con los 5 vecinos más cercanos **de otras provincias** (así el vecino no es de la misma zona hidrológica que el municipio que se oculta, como pasaría con los 722 reales, que suelen estar en zonas contiguas). Se compara con dos alternativas simples. Error medio absoluto en puntos porcentuales (más bajo = mejor).

In [3]:
con = d[COL_ACUIFERO].notna().to_numpy()
y = d.loc[con, COL_ACUIFERO].to_numpy()
prov = d.loc[con, 'cod_ine'].str[:2].to_numpy()
coords = np.radians(d.loc[con, ['lat', 'lon']].to_numpy())
_, idx = BallTree(coords, metric='haversine').query(coords, k=min(1000, len(y)))

pred = np.full(len(y), np.nan)
for i in range(len(y)):
    otros = [j for j in idx[i] if prov[j] != prov[i]][:VECINOS]
    if len(otros) == VECINOS:  # si no hay 5 vecinos de otras provincias entre los más cercanos, no se evalúa
        pred[i] = y[otros].mean()
valido = ~np.isnan(pred)
print(f'Municipios evaluados: {int(valido.sum())} de {len(y)}')
y, pred = y[valido], pred[valido]

comparacion = pd.Series({
    'poner 0 (sin dato = buen estado)': np.abs(y).mean(),
    'media de todos los municipios': np.abs(y - y.mean()).mean(),
    f'{VECINOS} vecinos más cercanos de otras provincias': np.abs(y - pred).mean(),
}).round(1).rename('error_medio_pp')
print(comparacion.to_string())
print('Correlación (Spearman) con el valor real:', round(float(pd.Series(y).corr(pd.Series(pred), method='spearman')), 2))

Municipios evaluados: 7410 de 7410
poner 0 (sin dato = buen estado)              16.5
media de todos los municipios                 26.1
5 vecinos más cercanos de otras provincias    12.8
Correlación (Spearman) con el valor real: 0.67


**Lectura:** la estimación con vecinos se equivoca **menos que la media de todos los municipios** (error de unos 26 puntos) y **algo menos que rellenar con 0** (unos 16 frente a unos 13); la mejora frente al 0 es moderada y el error sigue siendo apreciable. Es una aproximación, no un dato. Por eso en la app siempre se marca como `aproximado` y no se usa para afirmar que el agua de un municipio está bien o mal con seguridad.

La validación por provincia es **más exigente** que ocultar un municipio y estimarlo con sus vecinos inmediatos, que daría un error optimista.

> Los 722 municipios sin dato reciben, de media, un valor estimado bajo (≈ 6 %), por lo que para ellos rellenar con 0 daría un resultado parecido. Aun así, el 0 diría "buen estado" como si se hubiera medido, y la estimación con vecinos no.